In [ ]:
import pandas as pd
df = pd.read_csv("../data/cardio_dataset.csv", sep=";")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df["cardio"].value_counts()

In [ ]:
df["gender"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

In [ ]:
sns.countplot(x="cardio", data=df)
plt.title("Cardiovascular Disease Distribution")
plt.show()

In [ ]:
df["age_years"] = df["age"] / 365

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["age_years"], bins=30)
plt.title("Age Distribution")
plt.show()

In [ ]:
df = df.drop(columns=["id"])
df = df.drop(columns=["age"])

In [ ]:
df = df[(df["ap_hi"] > 50) & (df["ap_hi"] < 250)]
df = df[(df["ap_lo"] > 30) & (df["ap_lo"] < 200)]
df.shape

In [ ]:
plt.figure(figsize=(5,3))
sns.histplot(df["ap_hi"], bins=50)
plt.title("Systolic Blood Pressure")
plt.show()

plt.figure(figsize=(5,3))
sns.histplot(df["ap_lo"], bins=50)
plt.title("Diastolic Blood Pressure")
plt.show()

In [ ]:
df["height_m"] = df["height"] / 100
df["bmi"] = df["weight"] / (df["height_m"] ** 2)

In [ ]:
df = df.drop(columns=["height_m"])

In [ ]:
df = df[(df["bmi"] > 10) & (df["bmi"] < 60)]
df.shape

In [ ]:
df["bmi"].describe()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=False, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
X = df.drop(columns=["cardio"])
y = df["cardio"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_pred[:10]

In [ ]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

In [ ]:
rf_accuracy = accuracy_score(y_test, rf_pred)
print("Accuracy:", rf_accuracy)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, rf_pred))

XG Boost

In [ ]:
from xgboost import XGBClassifier
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric="logloss")

In [ ]:
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [ ]:
xgb_accuracy = accuracy_score(y_test, xgb_pred)
print("Accuracy:", xgb_accuracy)

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, xgb_pred))

In [ ]:
import xgboost as xgb

plt.figure(figsize=(10,6))
xgb.plot_importance(xgb_model, max_num_features=10)
plt.title("Top Important Features for Disease Prediction")
plt.show()

Model Optimization

In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.01, 0.1]
}

xgb = XGBClassifier(eval_metric="logloss")

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

In [ ]:
print(grid_search.best_params_)

In [ ]:
print(grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)

from sklearn.metrics import accuracy_score
best_accuracy = accuracy_score(y_test, y_pred_best)
print("Best XGBoost Accuracy:", best_accuracy)

In [ ]:
import joblib

joblib.dump(best_model, "../models/cardiovascular_model.pkl")